In [1]:
# Cell 1 — Star list (primary input)
# This is the only cell that needs editing to add a new star.
# displayName must resolve via the VSX "ident" lookup (e.g. "CY Aqr", "W UMa").
# objectId is a stable lowercase identifier derived from displayName unless overridden here.

stars = [
    {"displayName": "CY Aqr"},
    {"displayName": "YZ Boo"},
    {"displayName": "W UMa"},
]

In [2]:
# Cell 2 — Imports & config

import json
import re
import urllib.request
import urllib.parse
import pandas as pd

VSX_API_URL = "https://www.aavso.org/vsx/index.php"
OUTPUT_FILE = "mobile/vs.json"


In [3]:
# Cell 3 — Lookup maps

# IAU 3-letter abbreviation -> full constellation name (same table as dso_catalog.ipynb)
CONSTELLATION_NAMES = {
    "And": "Andromeda",       "Ant": "Antlia",          "Aps": "Apus",
    "Aqr": "Aquarius",        "Aql": "Aquila",           "Ara": "Ara",
    "Ari": "Aries",           "Aur": "Auriga",           "Boo": "Boötes",
    "Cae": "Caelum",          "Cam": "Camelopardalis",   "Cnc": "Cancer",
    "CVn": "Canes Venatici",  "CMa": "Canis Major",      "CMi": "Canis Minor",
    "Cap": "Capricornus",     "Car": "Carina",            "Cas": "Cassiopeia",
    "Cen": "Centaurus",       "Cep": "Cepheus",          "Cet": "Cetus",
    "Cha": "Chamaeleon",      "Cir": "Circinus",         "Col": "Columba",
    "Com": "Coma Berenices",  "CrA": "Corona Australis", "CrB": "Corona Borealis",
    "Crv": "Corvus",          "Crt": "Crater",            "Cru": "Crux",
    "Cyg": "Cygnus",          "Del": "Delphinus",        "Dor": "Dorado",
    "Dra": "Draco",           "Equ": "Equuleus",         "Eri": "Eridanus",
    "For": "Fornax",          "Gem": "Gemini",            "Gru": "Grus",
    "Her": "Hercules",        "Hor": "Horologium",       "Hya": "Hydra",
    "Hyi": "Hydrus",          "Ind": "Indus",             "Lac": "Lacerta",
    "Leo": "Leo",             "LMi": "Leo Minor",         "Lep": "Lepus",
    "Lib": "Libra",           "Lup": "Lupus",             "Lyn": "Lynx",
    "Lyr": "Lyra",            "Men": "Mensa",             "Mic": "Microscopium",
    "Mon": "Monoceros",       "Mus": "Musca",             "Nor": "Norma",
    "Oct": "Octans",          "Oph": "Ophiuchus",         "Ori": "Orion",
    "Pav": "Pavo",            "Peg": "Pegasus",           "Per": "Perseus",
    "Phe": "Phoenix",         "Pic": "Pictor",            "Psc": "Pisces",
    "PsA": "Piscis Austrinus","Pup": "Puppis",            "Pyx": "Pyxis",
    "Ret": "Reticulum",       "Sge": "Sagitta",           "Sgr": "Sagittarius",
    "Sco": "Scorpius",        "Scl": "Sculptor",          "Sct": "Scutum",
    "Se1": "Serpens Caput",   "Se2": "Serpens Cauda",     "Ser": "Serpens",
    "Sex": "Sextans",         "Tau": "Taurus",            "Tel": "Telescopium",
    "Tri": "Triangulum",      "TrA": "Triangulum Australe", "Tuc": "Tucana",
    "UMa": "Ursa Major",      "UMi": "Ursa Minor",        "Vel": "Vela",
    "Vir": "Virgo",           "Vol": "Volans",            "Vul": "Vulpecula",
}

# VariabilityType family -> whether the VSX Epoch field marks a MIN or MAX.
# Pulsators: epoch is conventionally epoch-of-maximum-light.
# Eclipsing binaries: epoch is conventionally epoch-of-primary-minimum.
# Any VariabilityType not listed here falls through to EPOCH_TYPE_OVERRIDES,
# and if still unresolved, the build step will flag it for a manual override.
EPOCH_TYPE_BY_VARTYPE_PREFIX = {
    "SXPHE": "MAX",
    "HADS":  "MAX",
    "DSCT":  "MAX",
    "RR":    "MAX",   # RRAB, RRC, RRD...
    "CEP":   "MAX",
    "EA":    "MIN",
    "EB":    "MIN",
    "EW":    "MIN",   # includes "EW/KW" contact binaries
}


In [4]:
# Cell 4 — Per-object overrides
# objectId -> field overrides, applied after the VSX lookup.
# Use this for a corrected objectId, a resolved variableEpochType ambiguity,
# or any field VSX doesn't return cleanly.

OBJECT_ID_OVERRIDES = {
    # "CY Aqr": "cyaqr",   # only needed if the default slugify produces something wrong
}

EPOCH_TYPE_OVERRIDES = {
    # "someobjectid": "MIN",
}

FIELD_OVERRIDES = {
    # "someobjectid": {"variableEpochJd": 2459000.123},
}


In [5]:
# Cell 5 — VSX lookup

def slugify_object_id(display_name):
    """Default objectId derivation: lowercase, strip non-alphanumerics."""
    return re.sub(r"[^a-z0-9]", "", display_name.lower())


def parse_mag(mag_str):
    """VSX magnitude fields look like \"10.42 V\" — take the leading numeric part."""
    if not mag_str:
        return None
    m = re.match(r"[-+]?[0-9.]+", mag_str.strip())
    return float(m.group(0)) if m else None


def epoch_type_for(vartype, object_id):
    if object_id in EPOCH_TYPE_OVERRIDES:
        return EPOCH_TYPE_OVERRIDES[object_id]
    for prefix, epoch_type in EPOCH_TYPE_BY_VARTYPE_PREFIX.items():
        if vartype and vartype.startswith(prefix):
            return epoch_type
    return None


def vsx_lookup(display_name):
    """Query the AAVSO VSX API for a single star by name. Returns a result dict or None."""
    params = urllib.parse.urlencode({"view": "api.object", "ident": display_name, "format": "json"})
    url = f"{VSX_API_URL}?{params}"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            data = json.loads(resp.read().decode("utf-8"))
    except Exception as e:
        print(f"  VSX error for {display_name!r}: {e}")
        return None

    obj = data.get("VSXObject")
    if not obj:
        return None

    object_id = OBJECT_ID_OVERRIDES.get(display_name, slugify_object_id(obj.get("Name", display_name)))
    vartype = obj.get("VariabilityType")

    try:
        ra = float(obj["RA2000"]) / 15.0   # VSX RA2000 is decimal degrees -> decimal hours
    except (KeyError, ValueError, TypeError):
        ra = None
    try:
        dec = float(obj["Declination2000"])
    except (KeyError, ValueError, TypeError):
        dec = None
    try:
        period = float(obj["Period"])
    except (KeyError, ValueError, TypeError):
        period = None
    try:
        epoch_jd = float(obj["Epoch"])
    except (KeyError, ValueError, TypeError):
        epoch_jd = None

    constellation_abbr = obj.get("Constellation")
    constellation = CONSTELLATION_NAMES.get(constellation_abbr, constellation_abbr)

    result = {
        "displayName":         obj.get("Name", display_name),
        "objectId":            object_id,
        "ra":                  ra,
        "dec":                 dec,
        "subType":             vartype,
        "constellation":       constellation,
        "magnitude":           parse_mag(obj.get("MaxMag")),
        "variablePeriodDays":  period,
        "variableEpochJd":     epoch_jd,
        "variableEpochType":   epoch_type_for(vartype, object_id),
    }

    if object_id in FIELD_OVERRIDES:
        result.update(FIELD_OVERRIDES[object_id])

    return result


lookup_results = []
for star in stars:
    print(f"Looking up {star['displayName']}...")
    result = vsx_lookup(star["displayName"])
    if result is None:
        print(f"  NOT FOUND: {star['displayName']}")
        continue
    lookup_results.append(result)

print(f"Resolved {len(lookup_results)} / {len(stars)} stars.")


Looking up CY Aqr...
Looking up YZ Boo...
Looking up W UMa...
Resolved 3 / 3 stars.


In [6]:
# Cell 6 — Build & validate DataFrame

df = pd.DataFrame(lookup_results)

errors = []

missing_coords = df[df["ra"].isna() | df["dec"].isna()]["objectId"].tolist()
if missing_coords:
    errors.append(f"Missing ra/dec for: {missing_coords}")

missing_period = df[df["variablePeriodDays"].isna()]["objectId"].tolist()
if missing_period:
    errors.append(f"Missing variablePeriodDays for: {missing_period}")

missing_epoch = df[df["variableEpochJd"].isna()]["objectId"].tolist()
if missing_epoch:
    errors.append(f"Missing variableEpochJd for: {missing_epoch}")

missing_epoch_type = df[df["variableEpochType"].isna()]["objectId"].tolist()
if missing_epoch_type:
    errors.append(
        f"Could not resolve variableEpochType for: {missing_epoch_type} "
        f"— add to EPOCH_TYPE_BY_VARTYPE_PREFIX or EPOCH_TYPE_OVERRIDES"
    )

dupes = df[df.duplicated("objectId")]["objectId"].tolist()
if dupes:
    errors.append(f"Duplicate objectIds: {dupes}")

if errors:
    raise ValueError("Validation failed:\n" + "\n".join(errors))

print(f"Validation passed. {len(df)} stars ready.")
df


Validation passed. 3 stars ready.


,displayName,objectId,ra,dec,subType,constellation,magnitude,variablePeriodDays,variableEpochJd,variableEpochType
0,CY Aqr,cyaqr,22.629958,1.53439,SXPHE,Aquarius,10.42,0.061038,2.460588e+06,MAX
1,YZ Boo,yzboo,15.401945,36.86683,HADS,Boötes,10.33,0.104092,2.457118e+06,MAX
2,W UMa,wuma,9.729297,55.95253,EW/KW,Ursa Major,7.75,0.333633,2.453762e+06,MIN


In [7]:
# Cell 7 — Serialize to vs.json

output_cols = [
    "displayName",
    "objectId",
    "ra",
    "dec",
    "subType",
    "constellation",
    "magnitude",
    "variablePeriodDays",
    "variableEpochJd",
    "variableEpochType",
]

out_df = df[output_cols].copy()
out_df["ra"] = out_df["ra"].round(7)
out_df.to_json(OUTPUT_FILE, orient="records", indent=2)
print(f"Written {len(out_df)} stars to {OUTPUT_FILE}")


Written 3 stars to mobile/vs.json
